# ラボ 2b: babyweight データセットの準備

**学習目標**

1. 環境をセットアップする
1. 出生データセット（natality dataset）を前処理する
1. 出生データセットをデータ拡張（augment）する
1. BigQuery で訓練テーブルと評価テーブルを作成する
1. BigQuery から GCS へ CSV 形式でデータをエクスポートする


## はじめに
このノートブックでは、生まれる前の赤ちゃんの体重を予測するモデルの開発と訓練に向けて、babyweight データセットを準備します。BigQuery を使用してデータの拡張と前処理を実行し、それらを AutoML Tables、BigQuery ML、および Cloud AI Platform で訓練される Keras モデルに使用します。

このラボでは、環境のセットアップ、プロジェクトデータセットの作成、出生データセットの前処理とデータ拡張、BigQuery での訓練・評価テーブルの作成、および BigQuery から GCS への CSV 形式でのデータのエクスポートを行います。

各学習目標は、この受講者用ラボノートブックの __#TODO__ に対応しています。まずこのノートブックを完了してから、[ソリューションノートブック](../solutions/1b_prepare_data_babyweight.ipynb) を確認するようにしてください。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートします。

In [ ]:
import os

from google.cloud import bigquery
%load_ext google.cloud.bigquery

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト ID を使用します。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

## ソースデータセット

データセットは [BigQuery](https://cloud.google.com/bigquery/) でホストされています。CDC の Natality データには 1969 年から 2008 年までの米国の出生に関する詳細が含まれており、一般公開されているデータセットであるため、GCP アカウントを持つ人なら誰でもアクセスできます。データセットにアクセスするには、[こちら](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) をクリックしてください。

出生データセットは約 1 億 3,800 万行と 31 列からなる比較的大きなデータセットですが、理解しやすい内容です。`weight_pounds` がターゲットであり、モデルを訓練して予測する連続値です。

## BigQuery データセットと Google Cloud Storage バケットの作成

BigQuery データセットは、BigQuery ML で構築されたテーブル、ビュー、モデルのコンテナです。以前のラボでまだ作成していない場合は、__babyweight__ という名前のデータセットを作成しましょう。プロジェクト用の GCS バケットについても同様に行います。

In [ ]:
%%bash

## babyweight の BigQuery データセットが存在しない場合は作成
datasetexists=$(bq ls -d | grep -w ) # TODO: データセット名を追加

if [ -n "$datasetexists" ]; then
    echo -e "BigQuery dataset already exists, let's not recreate it."

else
    echo "Creating BigQuery dataset titled: babyweight"
    
    bq --location=US mk --dataset \
        --description "Babyweight" \
        $PROJECT:# TODO: データセット名を追加
    echo "Here are your current datasets:"
    bq ls
fi
    
## GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
    
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "Here are your current buckets:"
    gcloud storage ls
fi

## 訓練データテーブルと評価データテーブルの作成

すでに一般公開されているデータセットがあるため、この未加工の入力データを使用して訓練データテーブルと評価データテーブルを簡単に作成できます。まず、列を `weight_pounds`、`is_male`、`mother_age`、`plurality`、`gestation_weeks` に限定し、簡単なフィルタリングと再現性のある分割のためのハッシュ列を追加したデータのサブセットを作成します。

* 注: 下記のテーブル作成コードのデータセットは、先ほど作成したもの（例: "babyweight"）です。

### ラボ課題 1: データセットの前処理とフィルタリング

訓練に適した形式でデータを取得するために、いくつかの前処理とフィルタリングを実行します。

前処理:
* `is_male` を `BOOL` から `STRING` にキャスト
* `plurality` を `INTEGER` から `STRING` にキャスト（`[1, 2, 3, 4, 5]` を `["Single(1)", "Twins(2)", "Triplets(3)", "Quadruplets(4)", "Quintuplets(5)"]` に変換）
* `year` と `month` に基づいてハッシュ化した `hashmonth` 列を追加

フィルタリング:
* `2000` 年より後のデータのみを対象とする
* 赤ちゃんの体重が `0` より大きいデータのみを対象とする
* 母親の年齢が `0` より大きいデータのみを対象とする
* 出生数（plurality）が `0` より大きいデータのみを対象とする
* 妊娠期間（gestation_weeks）が `0` より大きいデータのみを対象とする

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data AS
SELECT
    # TODO: 選択した生の特徴量と前処理された特徴量を追加
FROM
    publicdata.samples.natality
WHERE
    # TODO: フィルタを追加

### ラボ課題 2: 欠損データをシミュレートするためのデータセットの拡張

ここで、すべての性別情報を `Unknown` に設定し、単胎（single）以外のすべての出生の plurality を `Multiple(2+)` に設定することで、シミュレートした babyweight データでデータセットを拡張します。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_augmented_data AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    hashmonth
FROM
    babyweight.babyweight_data
UNION ALL
SELECT
    # TODO: 上記の指示に従って is_male と plurality を置換
FROM
    babyweight.babyweight_data

### ラボ課題 3: 拡張データセットを訓練セットと評価セットに分割

`hashmonth` を使用して剰余（modulo）演算を適用し、およそ 75/25 の train/eval 分割を取得します。

### 拡張データセットを訓練データセットに分割

#### **演習**: クエリを**実行**して訓練データテーブルを作成してください。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data_train AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks
FROM
    babyweight.babyweight_augmented_data
WHERE
    # TODO: hashmonth の剰余を取ってデータの約 75% にする

### 拡張データセットを評価データセットに分割

#### **演習**: クエリを**実行**して評価データテーブルを作成してください。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data_eval AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks
FROM
    babyweight.babyweight_augmented_data
WHERE
    # TODO: hashmonth の剰余を取ってデータの約 25% にする

## テーブル作成の確認

データセットと訓練データテーブルが作成されたことを確認します。

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

## ラボ課題 4: BigQuery から GCS の CSV へのエクスポート

BigQuery Python API を使用して、後で TensorFlow / Keras の訓練で使用するために、訓練テーブルと評価テーブルを CSV 形式で Google Cloud Storage にエクスポートします。上記で使用してきたデータセットを使用し、訓練データと評価データの両方に対してプロセスを繰り返します。

In [ ]:
# BigQuery クライアントオブジェクトを構築
client = bigquery.Client()

dataset_name = # TODO: データセット名を追加

# データセット参照オブジェクトを作成
dataset_ref = client.dataset(
    dataset_id=dataset_name, project=client.project)

# train テーブルと eval テーブルの両方をエクスポート
for step in [
    # TODO: train と eval をループ
]:
    destination_uri = os.path.join(
        "gs://", BUCKET, dataset_name, "data", "{}*.csv".format(step))
    table_name = "babyweight_data_{}".format(step)
    table_ref = dataset_ref.table(table_name)
    extract_job = client.extract_table(
        table_ref,
        destination_uri,
        # ロケーションは元のテーブルと一致している必要があります。
        location="US",
    )  # API リクエスト
    extract_job.result()  # ジョブの完了を待機

    print("Exported {}:{}.{} to {}".format(
        client.project, dataset_name, table_name, destination_uri))

## CSV 作成の確認

バケット内に CSV ファイルが正しく作成されたことを確認します。

In [ ]:
%%bash
gcloud storage ls gs://${BUCKET}/babyweight/data/*.csv

In [ ]:
%%bash
gcloud storage cat gs://${BUCKET}/babyweight/data/train000000000000.csv 2>/dev/null | head -5

In [ ]:
%%bash
gcloud storage cat gs://${BUCKET}/babyweight/data/eval000000000000.csv 2>/dev/null | head -5

## ラボのまとめ:
このラボでは、環境のセットアップ、BigQuery データセットの作成、出生データセットの前処理とデータ拡張、BigQuery での訓練テーブルと評価テーブルの作成、および BigQuery から GCS への CSV 形式でのデータのエクスポートを行いました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.